<a href="https://colab.research.google.com/github/Dexbar/dexter-dao-refi/blob/main/notebookedd8a7b577.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

dexbar_gemma_terminal_train_path = kagglehub.dataset_download('dexbar/gemma-terminal-train')
google_gemma_4_transformers_gemma_4_12b_it_2_path = kagglehub.model_download('google/gemma-4/Transformers/gemma-4-12b-it/2')

print('Data source import complete.')


KaggleApiHTTPError: 403 Client Error.

You don't have permission to access resource at URL: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDataset. Please make sure you are authenticated if you are trying to access a private resource or a resource requiring consent.

# 🚀 Gemma 4 12B - Entrenamiento LoRA para Comandos de Terminal Hardhat

## ⚠️ PASOS IMPORTANTES:
1. Asegúrate de tener GPU habilitada (Settings → Accelerator → GPU T4 x2)
2. Sube tu dataset como Kaggle Dataset antes de correr
3. Al terminar, usa **Save & Run All (Commit)** para guardar los pesos

In [ ]:
# Instalar dependencias
!pip install -q transformers peft accelerate bitsandbytes trl datasets kagglehub

In [ ]:
# Instalar versión actualizada de transformers
!pip install -q -U transformers
!pip install -q peft accelerate bitsandbytes trl datasets kagglehub

In [ ]:
import os
import json
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset

# ✅ RUTA DE SALIDA - DEBE ser /kaggle/working/ para que se guarde como output
OUTPUT_DIR = "/kaggle/working/gemma-lora-output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ Directorio de salida: {OUTPUT_DIR}")
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
model_path = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-12b-it/2"

In [ ]:
# Cargar tokenizer y modelo con cuantización 4-bit (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
model = prepare_model_for_kbit_training(model)
print("✅ Modelo cargado con cuantización 4-bit QLoRA")

In [ ]:
import os

# Encontrar la ruta exacta del modelo
print("Contenido de /kaggle/input/:")
for item in os.listdir('/kaggle/input'):
    print(f"  {item}")
    subpath = f"/kaggle/input/{item}"
    try:
        for sub in os.listdir(subpath):
            print(f"    {sub}")
    except:
        pass

In [ ]:
import os

# Ver la ruta completa del modelo
for root, dirs, files in os.walk('/kaggle/input/google'):
    for f in files:
        if f == 'config.json':
            print("✅ Modelo encontrado en:", root)
            break

In [ ]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f in ['config.json', 'model.safetensors', 'tokenizer.json']:
            print(root)
            break

In [ ]:
# Configuración LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cargar y formatear tu dataset terminal_train.jsonl
# El dataset tiene formato: {"prompt": "...", "command": "..."}

def format_for_gemma(example):
    """Convierte prompt+command al formato de chat de Gemma"""
    text = (
        f"<start_of_turn>user\n"
        f"{example['prompt']}<end_of_turn>\n"
        f"<start_of_turn>model\n"
        f"{example['command']}<end_of_turn>"
    )
    return {"text": text}

# Cargar desde /kaggle/input/ (después de subir el dataset a Kaggle)
# ⚠️ Cambia 'tu-dataset' por el nombre de tu dataset en Kaggle
DATA_PATH = "/kaggle/input/gemma-terminal-train/terminal_train.jsonl"

if os.path.exists(DATA_PATH):
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f if line.strip()]
    print(f"✅ Dataset cargado desde Kaggle: {len(data)} ejemplos")
else:
    # Fallback: datos de ejemplo si no se encuentra el archivo
    print("⚠️ No se encontró el dataset en Kaggle. Usando datos de ejemplo.")
    print("   Sube terminal_train.jsonl como Kaggle Dataset para usar tus datos reales.")
    data = [
        {"prompt": "compile the smart contracts", "command": "npx hardhat compile"},
        {"prompt": "compilar los contratos inteligentes", "command": "npx hardhat compile"},
        {"prompt": "run tests", "command": "npx hardhat test"},
        {"prompt": "correr las pruebas", "command": "npx hardhat test"},
        {"prompt": "deploy to localhost", "command": "npx hardhat run scripts/deploy.js --network localhost"},
    ]

dataset = Dataset.from_list(data).map(format_for_gemma)
print(f"\nEjemplo de formato:")
print(dataset[0]["text"])

In [ ]:
# Configuración de entrenamiento y entrenador
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=5,
    save_strategy="epoch",
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=512
)

print("✅ Iniciando entrenamiento...")
trainer.train()

In [ ]:
# ✅ GUARDAR EL LORA EN /kaggle/working/ (esto SÍ se guarda como output)
print("Guardando adaptador LoRA...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Verificar archivos guardados
print("\n✅ Archivos guardados:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / (1024*1024)
    print(f"  📄 {f} ({size:.2f} MB)")

print("\n🎉 ¡Entrenamiento completado!")
print("👉 Ahora haz clic en 'Save & Run All (Commit)' para guardar los outputs.")